In [1]:
!pip install transformers datasets torchmetrics pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.4/926.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 23.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AdamW, get_scheduler
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchmetrics.text.rouge import ROUGEScore
import torch
from torch import nn
import pytorch_lightning as pl
import numpy as np
import random

In [3]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
from datasets import DatasetDict
raw_datasets = DatasetDict({
    "train": load_dataset("json", data_files="/content/train-open.json", split="train"),
    "validation": load_dataset("json", data_files="/content/val-open.json", split="train"),
    "test": load_dataset("json", data_files="/content/test-open.json", split="train"),
})
print(raw_datasets)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question_id', 'answer_id', 'question', 'answer'],
        num_rows: 58676
    })
    validation: Dataset({
        features: ['question_id', 'answer_id', 'question', 'answer'],
        num_rows: 12715
    })
    test: Dataset({
        features: ['question_id', 'answer_id', 'question', 'answer'],
        num_rows: 12592
    })
})


In [5]:
MODEL_NAME = "aubmindlab/aragpt2-base"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # Set pad_token to eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

In [6]:
def preprocess_function(examples):
    inputs = ["question: " + q for q in examples["question"]]
    targets = examples["answer"]
    inputs = tokenizer(inputs, truncation=True, padding="max_length", max_length=512)
    targets = tokenizer(targets, truncation=True, padding="max_length", max_length=512)

    inputs["labels"] = targets["input_ids"]  # Ensure labels align with input_ids
    return inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/58676 [00:00<?, ? examples/s]

Map:   0%|          | 0/12715 [00:00<?, ? examples/s]

Map:   0%|          | 0/12592 [00:00<?, ? examples/s]

In [7]:
def collate_fn(batch):
    input_ids = pad_sequence([torch.tensor(x["input_ids"]) for x in batch], batch_first=True, padding_value=tokenizer.pad_token_id)
    labels = pad_sequence([torch.tensor(x["labels"]) for x in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "labels": labels}

train_dataloader = DataLoader(tokenized_datasets["train"], batch_size=4, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(tokenized_datasets["validation"], batch_size=4, collate_fn=collate_fn)

### Cell 8: Define Optimizer and Scheduler
optimizer = AdamW(model.parameters(), lr=5e-5)
scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=100, num_training_steps=len(train_dataloader) * 3
)


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [8]:
class DifferentialAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.WQ = nn.Linear(d_model, 2 * d_model)
        self.WK = nn.Linear(d_model, 2 * d_model)
        self.WV = nn.Linear(d_model, 2 * d_model)
        self.lambda_q1 = nn.Parameter(torch.rand(d_model // n_heads))
        self.lambda_k1 = nn.Parameter(torch.rand(d_model // n_heads))
        self.lambda_q2 = nn.Parameter(torch.rand(d_model // n_heads))
        self.lambda_k2 = nn.Parameter(torch.rand(d_model // n_heads))
        self.lambda_init = 0.8

    def forward(self, X):
        Q1, Q2 = torch.chunk(self.WQ(X), 2, dim=-1)
        K1, K2 = torch.chunk(self.WK(X), 2, dim=-1)
        V = self.WV(X)

        lambda_val = torch.exp(self.lambda_q1 @ self.lambda_k1.T) - torch.exp(self.lambda_q2 @ self.lambda_k2.T) + self.lambda_init

        attention_1 = torch.softmax(Q1 @ K1.transpose(-2, -1) / np.sqrt(Q1.size(-1)), dim=-1)
        attention_2 = torch.softmax(Q2 @ K2.transpose(-2, -1) / np.sqrt(Q2.size(-1)), dim=-1)

        diff_attention = attention_1 - lambda_val * attention_2
        return diff_attention @ V

In [9]:
def train_epoch(model, dataloader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        inputs = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [10]:
def evaluate_epoch(model, dataloader):
    model.eval()
    total_loss = 0
    rouge = ROUGEScore()
    with torch.no_grad():
        for batch in dataloader:
            inputs = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=inputs, labels=labels)
            total_loss += outputs.loss.item()
    return total_loss / len(dataloader)

In [ ]:
EPOCHS = 3
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_dataloader, optimizer, scheduler)
    val_loss = evaluate_epoch(model, val_dataloader)
    print(f"Epoch {epoch + 1}: Train Loss = {train_loss:.4f}, Validation Loss = {val_loss:.4f}")

In [ ]:
model.save_pretrained("./fine_tuned_gpt2")
tokenizer.save_pretrained("./fine_tuned_gpt2")

In [ ]:
def test_model(question):
    inputs = tokenizer("question: " + question, return_tensors="pt", padding=True).to(device)
    outputs = model.generate(inputs["input_ids"], max_length=50)
    print("Answer:", tokenizer.decode(outputs[0], skip_special_tokens=True))

test_question = "What is the capital of Morocco?"
test_model(test_question)